In [ ]:
'''
To run this shole jupyter notebook as a python script follow this:

(1) activate conda environemt
(2) go to where this notebook is located in your computer
(3) use `python` to enter python with in your shell/terminal
(4) follow the above syntax

from json import load
filename = 'compute_ICA_matrices.ipynb'
with open(filename) as fp:
    nb = load(fp)

for cell in nb['cells']:
    if cell['cell_type'] == 'code':
        source = ''.join(line for line in cell['source'] if not line.startswith('%'))
        exec(source, globals(), locals())
'''

"\nTo run this shole jupyter notebook as a python script follow this:\n\n(1) activate conda environemt\n(2) go to where this notebook is located in your computer\n(3) use `python` to enter python with in your shell/terminal\n(4) follow the above syntax\n\n\nfrom json import load\nfilename = 'compute_ICA_matrices.ipynb'\nwith open(filename) as fp:\n    nb = load(fp)\n\nfor cell in nb['cells']:\n    if cell['cell_type'] == 'code':\n        source = ''.join(line for line in cell['source'] if not line.startswith('%'))\n        exec(source, globals(), locals())\n"

In [2]:
import os
import pandas as pd
import numpy as np

In [ ]:
# make correlation matrix based on ICA timeseries for maps
if "/Users/snaranjo" in os.getcwd():
    local_pc_flag = "/Users/snaranjo/Desktop/neurotranslate/mount_point"
else:
    local_pc_flag=""

chpc_root=local_pc_flag+"/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch"

choose_ICA_or_INFOMAP="INFOMAP" #ICA or INFOMAP
if choose_ICA_or_INFOMAP == "ICA":
    print("Picked ICA. Using ICA paths and dr data")
    maps = 15
    corr_type = "partial" # full or partial
    path_to_dr_timeseries=f"{chpc_root}/NeuroTranslate/generate_ICA/ABCD_ICA/ICAd15/groupICA15.dr/dr_output"
    output_path=f"{chpc_root}/NeuroTranslate/brain_reps_datasets/ABCD/maps_and_netmats/ICA_netmats/individual_netmats_{corr_type}"
    get_subject_list=pd.read_csv(chpc_root+"/NeuroTranslate/surf2netmat/utils/subj_ids/ABCD_full_ID_site.csv",header=None)[0].values.tolist()[1:] #need this to skip header of session
    csv_file = "%s/dr_%s/timecourse.csv"
elif choose_ICA_or_INFOMAP == "INFOMAP":
    print("Picked INFOMAP. Using INFOMAP paths and dr data")
    maps = 20
    corr_type= "partial" #full or partial
    prior_type="spatial"
    path_to_dr_timeseries=f"{chpc_root}/NeuroTranslate/brain_reps_datasets/infomap_prior_ABCDdr/dr_infomap_priors/ABCD_infomap20_no_smooth/{prior_type}"
    output_path=f"{chpc_root}/NeuroTranslate/brain_reps_datasets/infomap_prior_ABCDdr/maps_and_netmats/{prior_type}_netmats/individual_netmats_{corr_type}"
    get_subject_list=pd.read_csv(chpc_root+"/NeuroTranslate/surf2netmat/utils/dr_scripts/neurotranslate_abcd_subject_list.txt", header=None)[0].values.tolist()
    csv_file = "%s/dr_sub-%s/timecourse.csv" 
    
#make output pathdir
if not os.path.isdir(output_path):
    os.makedirs(output_path)

print(f"Checking correct subj list. N={len(get_subject_list)}")
get_subject_list_short = get_subject_list #option to do less or all subjects equal means all subjects
#subject loop
sub_skipped=[]
sub_bad_data=[]
sub_goodenough_data=[]
rs, cs = np.tril_indices(maps, k=-1) #only upper triangle to predict
# init_maps2netmat_data = np.zeros((len(get_subject_list_short), len(rs))) #can choose either rs or cs same thing
for subii, subID in enumerate(get_subject_list_short):
    if subii % 1000 == 0:
        print(f"Getting CSV of timeseries for subject:{subii}.")
    
    csv_file_specific = csv_file % (path_to_dr_timeseries, subID)    
    try:
        get_timeseries = pd.read_csv(csv_file_specific, header=None, sep=' ')
    except Exception:
        print(f"Subject {subID} does not have a dr output. Maybe cause subject lists are different?")
        sub_skipped.append(subID)
        continue

    # sometimes loading these csv files needs me to make columns into NaNs to seperate the maps, so dropping those
    get_timeseries = (get_timeseries.dropna(axis=1)).to_numpy()
    # print(get_timeseries.shape)
    #make into maps X time shape if needed
    d1, d2 = get_timeseries.shape
    if d1 > d2:
        get_timeseries = get_timeseries.T #make to be 15/20 (ICA/INFOMAP) by timepoints
    
    #if for some reason that subject has less timepoints than maps, skip em
    if get_timeseries.shape[0] < maps: #if d1 is less than maps, then above check fails cause time<maps. Save this list for later
        #force_to_be_mapsXtime 
        sub_bad_data.append(subID)
        get_timeseries = get_timeseries.T
    
    assert get_timeseries.shape[0] == maps # asserts to make sure corrmat is the correct one
    if get_timeseries.shape[1] > 300:
        sub_goodenough_data.append(subID)

    if corr_type == "full":
        get_lowertri = np.corrcoef(get_timeseries)[rs,cs]
    elif corr_type == "partial":
        # expects TIMExNODES
        rho=0.1
        n    = maps #data.shape[1]
        # print(f"time series shape-{get_timeseries.shape}")
        corr = np.cov(get_timeseries)
        # Regularize the covariance matrix
        corr = corr / np.sqrt(np.mean(corr.diagonal(0)**2))
        corr = np.linalg.inv(corr + (rho * np.eye(n)))
        # Compute partial correlations
        corr = -corr
        diags = np.sqrt(np.abs(corr.diagonal(0)))
        diags = np.tile(diags, (1, n)).reshape((n, n))
        corr  = (corr / diags.T) / diags
        get_lowertri = corr[rs,cs]
    else:
        raise ValueError("corrtype not recognized.")        
    
    np.save(f"{output_path}/{subID}_{choose_ICA_or_INFOMAP}_netmat.npy", get_lowertri) #save each subjects separately, better because subejct lists can change.

df = pd.DataFrame({
    "subID": sub_bad_data
})
df_good_enough = pd.DataFrame({
    "subID": sub_goodenough_data
})

df.to_csv(f"{output_path}/{choose_ICA_or_INFOMAP}_netmats_badsubs.csv")
df_good_enough.to_csv(f"{output_path}/{choose_ICA_or_INFOMAP}_netmats_goodenoughsubs.csv")

# # viz to check and did and looks good.
# import matplotlib.pyplot as plt
# plt.imshow(make_netmat)

Picked ICA. Using ICA paths and dr data
Checking correct subj list. N=8673
Getting CSV of timeseries for subject:0.
time series shape-(15, 1532)
time series shape-(15, 1520)
time series shape-(15, 1532)
time series shape-(15, 1532)
time series shape-(15, 1532)
time series shape-(15, 1532)
time series shape-(15, 1532)
time series shape-(15, 1532)
time series shape-(15, 1532)
time series shape-(15, 1149)
